<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Working with Managed Tables in Unity Catalog</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 2.2 Demo Interoperability and Performance Benefits of Managed Tables

This demo uses TPC-DS at scale factor 1000 (~2.8 billion rows of `store_sales`) to show the performance impact of Liquid Clustering, then creates managed Iceberg and Delta+UniForm copies of the same data and verifies them through PyIceberg as a stand-in for any external Iceberg engine.

## Learning Objectives

By the end of this demonstration, you will be able to:
- Quantify the performance impact of Liquid Clustering on selective queries
- Create managed Iceberg and Delta+UniForm tables with Liquid Clustering
- Confirm cross-format access by reading both from an external Iceberg client (PyIceberg)
- Recognize what plain Unity Catalog managed tables expose to external Iceberg readers - and what they don't

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2272B4">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Before starting this notebook, make sure you ran the **0 - Required Setup** notebook once to set up the course data.
  </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-2

## A. Performance Comparison - Clustered vs Unclustered

Liquid Clustering only earns its keep when the dataset is large enough that the planner can skip files. Two reference tables - <code>store_sales_unclustered</code> and <code>store_sales_clustered</code> - have been pre-built by <code>1 - Instructor Demo Setup</code>. The clustered copy is laid out by Liquid Clustering on <code>(ss_sold_date_sk, ss_item_sk)</code>. We run the same selective query against both and compare the scan stats.

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #00695c; font-size: 1.1em;">What to Watch</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The headline metrics are <b>files read</b> and <b>bytes read</b> - open the query profile after each query and compare. Wall-clock matters too but is noisier on serverless.</p>
        </div>
    </div>
</div>

### A1. Build the Unclustered Table - Reference Only

The unclustered baseline table is materialized ahead of class by `1 - Instructor Demo Setup` so the demo does not stall on a multi-billion-row CTAS during the session. The source is partitioned by date, so an explicit `ORDER BY rand()` shuffle is applied to produce a true unclustered baseline - otherwise a date-range filter would prune almost everything before clustering even mattered. The DDL below is shown for reference only - it has already been executed.

<div class="code-block" data-language="sql">
-- Unclustered reference copy of TPC-DS store_sales (~2.8B rows)
-- Shuffled with ORDER BY rand() so the baseline is genuinely unclustered
CREATE OR REPLACE TABLE store_sales_unclustered AS
SELECT * FROM samples.tpcds_sf1000.store_sales
ORDER BY rand();
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
    });
})();
</script>

### A2. Baseline: Query the Unclustered Table

A selective filter on a 7-day window and a narrow item range. Open the query profile and note **bytes read** and **files read**.

In [0]:
-- Selective query against the 2.8B-row unclustered table
-- Open the query profile -> note bytes read and files read
SELECT ss_item_sk,
       COUNT(*)              AS lines,
       SUM(ss_quantity)      AS total_qty,
       AVG(ss_sales_price)   AS avg_price
FROM store_sales_unclustered
WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451186  -- 7-day window
  AND ss_item_sk      BETWEEN 1000   AND 1010    -- 11 items
GROUP BY ss_item_sk
ORDER BY ss_item_sk;

### A3. Build the Clustered Table - Reference Only

The clustered comparison table is materialized ahead of class by `1 - Instructor Demo Setup` so the demo does not stall on a multi-billion-row CTAS during the session. The DDL below is shown for reference only - it has already been executed.

<div class="code-block" data-language="sql">
-- Liquid Clustering on the columns we filter on
CREATE OR REPLACE TABLE store_sales_clustered
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
AS SELECT * FROM store_sales_unclustered;
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
    });
})();
</script>

### A4. Same Query Against the Clustered Table

Run the same query. The result is identical, but the planner now skips the vast majority of files. Compare bytes read and files read against the baseline above.

In [0]:
-- Same selective query against the clustered table
-- Compare bytes read / files read against the unclustered run above
SELECT ss_item_sk,
       COUNT(*)              AS lines,
       SUM(ss_quantity)      AS total_qty,
       AVG(ss_sales_price)   AS avg_price
FROM store_sales_clustered
WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451186
  AND ss_item_sk      BETWEEN 1000   AND 1010
GROUP BY ss_item_sk
ORDER BY ss_item_sk;

### A5. Files and Layout Comparison

`DESCRIBE DETAIL` exposes the physical layout of each table - file count, total size, clustering columns. Pulling those into Python lets us cherry-pick the fields that matter and lay them out side-by-side.

In [0]:
%python
import pandas as pd

def describe_detail(table: str) -> dict:
    return spark.sql(f"DESCRIBE DETAIL {table}").toPandas().iloc[0].to_dict()

def cluster_cols(detail: dict):
    cc = detail.get("clusteringColumns")
    if cc is None:
        return []
    # clusteringColumns comes back as a numpy array; convert to plain list
    return list(cc)

unc = describe_detail("store_sales_unclustered")
clu = describe_detail("store_sales_clustered")

cmp = pd.DataFrame([
    {
        "table":             "store_sales_unclustered",
        "format":            unc["format"],
        "numFiles":          unc["numFiles"],
        "sizeInBytes":       unc["sizeInBytes"],
        "clusteringColumns": cluster_cols(unc),
    },
    {
        "table":             "store_sales_clustered",
        "format":            clu["format"],
        "numFiles":          clu["numFiles"],
        "sizeInBytes":       clu["sizeInBytes"],
        "clusteringColumns": cluster_cols(clu),
    },
])
display(cmp)

### A6. Per-Query File Pruning - How Many Files Did We Actually Read?

`DESCRIBE DETAIL` tells us how many files each table has in total. To find out how many of those files the engine *actually opened* for a given query, we count the distinct `_metadata.file_path` values returned by the same `WHERE` clause - this is the file path of every record that was scanned.

The difference is **files pruned** - the entire point of Liquid Clustering. The clustered table should prune almost everything; the unclustered table should prune almost nothing.

In [0]:
%python
import pandas as pd

PROBE_TEMPLATE = """
SELECT COUNT(DISTINCT _metadata.file_path) AS files_read
FROM {table}
WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451186
  AND ss_item_sk      BETWEEN 1000   AND 1010
"""

def total_files(table):
    return spark.sql(f"DESCRIBE DETAIL {table}").toPandas().iloc[0]["numFiles"]

def files_read(table):
    return spark.sql(PROBE_TEMPLATE.format(table=table)).collect()[0][0]

def stats(label, table):
    total = total_files(table)
    used  = files_read(table)
    pruned = total - used
    return {
        "label":        label,
        "table":        table,
        "total_files":  total,
        "files_read":   used,
        "files_pruned": pruned,
        "pct_pruned":   round(100.0 * pruned / total, 2) if total else 0.0,
    }

display(pd.DataFrame([
    stats("unclustered", "store_sales_unclustered"),
    stats("clustered",   "store_sales_clustered"),
]))

## B. Cross-Format Tables - Managed Iceberg and Delta + UniForm

Step 3 needs Iceberg-visible tables to verify cross-format access from PyIceberg. We CTAS a 1M-row sample (not the full 2.8B rows) into a managed Iceberg table and a Delta + UniForm table - both clustered on the same keys as Step 1. The volume here is just enough to demonstrate cross-format readability; the perf characteristics were already established in Step 1.

### B1. Managed Iceberg

Native managed Iceberg. Cluster on the same keys. Iceberg in UC does not currently support deletion vectors or row tracking, so we explicitly disable those (Liquid Clustering enables them by default).

In [0]:
-- Managed Iceberg with Liquid Clustering
-- 1M rows is plenty to demonstrate cross-format access from PyIceberg in Step 3 -
-- the perf characteristics were established in Step 1 against the full 2.8B-row tables.
CREATE OR REPLACE TABLE store_sales_iceberg
USING ICEBERG
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'false',
  'delta.enableRowTracking'     = 'false'
)
AS SELECT * FROM store_sales_clustered LIMIT 1000000; -- smaller to keep this step fast

SELECT
  t.data_source_format AS format,
  t.table_type,
  (SELECT COUNT(*) FROM store_sales_iceberg) AS row_count
FROM system.information_schema.tables t
WHERE t.table_catalog = current_catalog()
  AND t.table_schema  = current_schema()
  AND t.table_name    = 'store_sales_iceberg';

### B2. Delta + UniForm

Delta as the primary format with UniForm metadata generated alongside it, so external Iceberg clients can read it without copying data.

In [0]:
-- Delta + UniForm (Iceberg-readable Delta) - 1M rows for the cross-format demo
CREATE OR REPLACE TABLE store_sales_delta_uniform
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
TBLPROPERTIES (
  'delta.enableIcebergCompatV3'          = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
AS SELECT * FROM store_sales_clustered LIMIT 1000000; -- smaller to keep this step fast

SHOW TBLPROPERTIES store_sales_delta_uniform;

## C. Verify Cross-Format Access with PyIceberg

The real test of UniForm is whether an external Iceberg client can actually read the table. PyIceberg connects to Unity Catalog through the Iceberg REST endpoint, so any UC table that exposes Iceberg metadata - native Iceberg or Delta+UniForm - should be loadable. A plain UC managed table without UniForm should not be.

In [0]:
-- For comparison, a plain managed UC managed table (no UniForm) clustered on the same keys.
-- We expect PyIceberg to FAIL to load this one - that is the point.
CREATE OR REPLACE TABLE store_sales_delta_plain
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
AS SELECT * FROM store_sales_clustered LIMIT 1000000; -- smaller to keep this step fast

In [0]:
%python
%pip install pyiceberg[pyarrow] --quiet
dbutils.library.restartPython()

In [0]:
%python
from pyiceberg.catalog import load_catalog

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
warehouse = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
schema    = spark.sql("SELECT current_schema()  AS s").collect()[0]["s"]

rest_catalog = load_catalog(
    "unity",
    **{
        "type": "rest",
        "uri": f"https://{workspace_url}/api/2.1/unity-catalog/iceberg-rest",
        "token": token,
        "warehouse": warehouse,
    },
)

print(f"Connected to UC Iceberg REST catalog: {warehouse}.{schema}")

In [0]:
%python
def try_load(table_name: str, expectation: str):
    fqn = f"{schema}.{table_name}"
    try:
        tbl = rest_catalog.load_table(fqn)
        n_cols = len(tbl.schema().fields)
        n_snapshots = len(tbl.metadata.snapshots)
        outcome = f"OK   (loaded as Iceberg: {n_cols} cols, {n_snapshots} snapshot(s))"
    except Exception as e:
        outcome = f"FAIL ({type(e).__name__}: {str(e)[:80]})"
    print(f"  {expectation:9s} -> {outcome:60s} {fqn}")

print("Loading UC tables through the Iceberg REST catalog:")
print()
try_load("store_sales_iceberg",       expectation="expect OK")    # native managed Iceberg
try_load("store_sales_delta_uniform", expectation="expect OK")    # Delta + UniForm
try_load("store_sales_delta_plain",   expectation="expect FAIL")  # plain Delta - not visible via Iceberg REST

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What This Confirms</strong>
            <p style="margin: 8px 0 0 0; color: #333;">PyIceberg stands in for any external Iceberg engine - Snowflake, Trino, Flink, Spark on EMR. <b>Native Iceberg</b> and <b>Delta + UniForm</b> tables are visible through the UC Iceberg REST endpoint without copying data. Plain UC managed tables are not, which is exactly why UniForm exists.</p>
        </div>
    </div>
</div>

## Key Takeaways

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">Demo Summary</strong>
            <ul style="margin: 8px 0 0 16px; color: #333"><li><strong>Liquid Clustering</strong> turns selective queries on multi-billion-row tables into selective scans - bytes read drops by orders of magnitude</li><li><strong>UniForm</strong> generates Iceberg metadata alongside Delta so external Iceberg clients can read UC managed tables without copying data</li><li>Managed <strong>Iceberg</strong> and Delta + UniForm tables are both visible through the UC Iceberg REST endpoint - plain Delta is not</li><li>The <code>store_sales_iceberg</code> and <code>store_sales_delta_uniform</code> tables you just created are the source for the rest of this course - Modules 3, 4, and 5 build on them</li></ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>